In this demo, we set up an agent to emulate the behavior of model in the fingpt-forecaster project with AutoGen, which takes a company's ticker symbol, recent basic financials and market news as input and predict its stock movements.

For detail of the original project, check out  [FinGPT-Forecaster](https://github.com/AI4Finance-Foundation/FinGPT/tree/master/fingpt/FinGPT_Forecaster)!  🔥[Demo](https://huggingface.co/spaces/FinGPT/FinGPT-Forecaster), [Medium Blog](https://medium.datadriveninvestor.com/introducing-fingpt-forecaster-the-future-of-robo-advisory-services-50add34e3d3c) & [Model](https://huggingface.co/FinGPT/fingpt-forecaster_dow30_llama2-7b_lora) on Huggingface🤗!

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-test123"

In [1]:
print("Jupyter working")

Jupyter working


In [2]:
import finrobot
print(finrobot.__file__)

C:\Users\KIIT0001\miniconda3\envs\finrobot\lib\site-packages\finrobot\__init__.py


In [1]:
from finrobot.agents.workflow import SingleAssistant
print("FinRobot loaded")

FinRobot loaded


In [2]:
import time

start = time.time()

from finrobot.agents.workflow import SingleAssistant

print("FinRobot import finished")
print("Time:", time.time() - start)

FinRobot import finished
Time: 0.0009515285491943359


In [4]:
import warnings
warnings.filterwarnings("ignore")

import os
from finrobot.agents.workflow import SingleAssistant

# Minimal config
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    ],
    "timeout": 60,
}

# Create agent
assistant = SingleAssistant(
    {"name": "Market_Analyst", "description": "Stock analyst"},
    llm_config,
    human_input_mode="NEVER"
)

print("FinRobot agent initialized")

[autogen.oai.client: 03-10 09:41:09] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
FinRobot agent initialized


In [6]:
# ==========================================
# FINROBOT LLM INTRADAY BACKTEST (MULTI STOCK)
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------------
# INPUT LISTS (10 STOCKS)
# -----------------------------
symbols = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS",
    "LTIM.NS",
    "MPHASIS.NS",
    "PERSISTENT.NS",
    "COFORGE.NS"
]

# 10 timestamps
timestamps = [
    "2026-02-20 10:00:00",
    "2026-02-20 10:15:00",
    "2026-02-20 10:30:00",
    "2026-02-20 10:45:00",
    "2026-02-20 11:00:00",
    "2026-02-20 11:15:00",
    "2026-02-20 11:30:00",
    "2026-02-20 11:45:00",
    "2026-02-20 12:00:00",
    "2026-02-20 12:15:00"
]

# Store results
results = []

# -----------------------------
# LOOP OVER STOCKS & TIMES
# -----------------------------
for symbol, ts in zip(symbols, timestamps):

    target_datetime = pd.Timestamp(ts, tz="Asia/Kolkata")

    # Round to 15 min
    minute = (target_datetime.minute // 15) * 15
    target_datetime = target_datetime.replace(minute=minute, second=0)

    print(f"\nProcessing {symbol} at {target_datetime}")

    # -----------------------------
    # DOWNLOAD DATA
    # -----------------------------
    df = yf.download(symbol, period="60d", interval="15m", progress=False)

    if df.empty:
        print("No data")
        continue

    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert("Asia/Kolkata")

    if target_datetime not in df.index:
        print("Timestamp not found")
        continue

    # -----------------------------
    # HISTORY
    # -----------------------------
    history_df = df[df.index < target_datetime].tail(7 * 25)

    current_price = float(df.loc[target_datetime]["Close"])

    history_text = history_df[
        ["Open", "High", "Low", "Close", "Volume"]
    ].to_string()

    # -----------------------------
    # PROMPT
    # -----------------------------
    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}
Prediction Time: {target_datetime}

Past 7 days 15-minute OHLCV data:

{history_text}

Predict whether NEXT candle will move UP or DOWN.

Respond:
Prediction: UP or DOWN
Reason: short explanation
"""

    response = assistant.chat(prompt)

    # -----------------------------
    # PREDICTION
    # -----------------------------
    if response and "UP" in response.upper():
        predicted_direction = "UP"

    elif response and "DOWN" in response.upper():
        predicted_direction = "DOWN"

    else:
        last_close = float(history_df["Close"].iloc[-1])
        prev_close = float(history_df["Close"].iloc[-2])

        predicted_direction = "UP" if last_close > prev_close else "DOWN"

    # Predicted price
    predicted_price = (
        current_price * 1.002
        if predicted_direction == "UP"
        else current_price * 0.998
    )

    # -----------------------------
    # ACTUAL RESULT
    # -----------------------------
    next_index = df.index.get_loc(target_datetime) + 1

    if next_index >= len(df):
        continue

    actual_price = float(df.iloc[next_index]["Close"])

    actual_direction = "UP" if actual_price > current_price else "DOWN"

    # -----------------------------
    # ERROR %
    # -----------------------------
    error_percentage = abs(predicted_price - actual_price) / actual_price * 100

    # Store result
    results.append({
        "Stock": symbol,
        "Time": target_datetime,
        "Curr Price": current_price,
        "Pred Price": predicted_price,
        "Act Price": actual_price,
        "Pred Dir": predicted_direction,
        "Act Dir": actual_direction,
        "Error %": error_percentage
    })

# -----------------------------
# RESULT TABLE
# -----------------------------
results_df = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

print("\n====================================")
print("FINAL BACKTEST RESULTS")
print("====================================")
print(results_df)

# -----------------------------
# AVERAGE ERROR
# -----------------------------
if not results_df.empty:
    print("\nAverage Error %:", results_df["Error %"].mean())


Processing TCS.NS at 2026-02-20 10:00:00+05:30
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS
Prediction Time: 2026-02-20 10:00:00+05:30

Past 7 days 15-minute OHLCV data:

Price                             Open         High          Low        Close   Volume
Ticker                          TCS.NS       TCS.NS       TCS.NS       TCS.NS   TCS.NS
Datetime                                                                              
2026-02-11 10:00:00+05:30  2961.500000  2964.199951  2954.300049  2958.300049   125647
2026-02-11 10:15:00+05:30  2959.300049  2970.800049  2958.699951  2965.699951   135333
2026-02-11 10:30:00+05:30  2965.199951  2974.899902  2964.300049  2971.399902    86745
2026-02-11 10:45:00+05:30  2971.800049  2973.000000  2968.000000  2969.100098    60139
2026-02-11 11:00:00+05:30  2969.100098  2971.000000  2958.300049  2960.100098    92231
2026-02-11 11:15:00+05:30  2960.399902  2962.000000  2957.199951  2957.800049    65

After importing all the necessary packages and functions, we first instantiate a market analysis assistant and a user proxy using the agent classes defined by AutoGen. We also need the config for OpenAI & Finnhub here. 
- for openai configuration, rename OAI_CONFIG_LIST_sample to OAI_CONFIG_LIST and replace the api keys
- for finnhub configuration, rename config_api_keys_sample to config_api_keys and replace the api keys

Then we register our predefined functions with the userproxy.

Following our implementation with the fingpt-forecaster, we combined news/financials calls to the Finnhub API and stock data calls to the YFinance API.

Start the conversation, asking the agent to call all the functions, and see how it aggregates all the information and leads to the conclusion.

In [5]:
import os
os.environ["OPENAI_API_KEY"] = "sk-your-real-key-here"
print("Key loaded")

Key loaded


In [3]:
from finrobot.agents.workflow import SingleAssistant

llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)

print("FinRobot Agent Ready ✅")

[autogen.oai.client: 03-09 23:20:02] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-09 23:20:02] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-09 23:20:02] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-09 23:20:02] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-09 23:20:03] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
FinRobot Agent Ready ✅


In [7]:
import os
os.environ["OPENAI_API_KEY"] = "sk-your-real-key-here"
print("Key loaded")

Key loaded


In [13]:
# ==========================================
# FINROBOT FORWARD TEST (MULTI STOCK)
# ==========================================

import os
import yfinance as yf
import pandas as pd
import numpy as np
import re
from datetime import datetime
import pytz
import json
from finrobot.agents.workflow import SingleAssistant


# -----------------------------
# LLM CONFIG
# -----------------------------

llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# -----------------------------
# SETTINGS
# -----------------------------

stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS",
    "MPHASIS.NS",
    "PERSISTENT.NS",
    "COFORGE.NS",
    "OFSS.NS"
]

interval = "5m"
forecast_horizons = [15, 30, 60]

ist = pytz.timezone("Asia/Kolkata")

forward_timestamp = datetime.now(ist).strftime("%Y-%m-%d %H:%M:%S")

forward_results = []

print("Forward Test Timestamp:", forward_timestamp)


# -----------------------------
# LOOP THROUGH STOCKS
# -----------------------------

for symbol in stocks:

    print("\nRunning Forward Test:", symbol)

    df = yf.download(
        symbol,
        period="5d",
        interval=interval,
        progress=False
    )

    if df.empty:
        continue

    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert("Asia/Kolkata")

    summary = {}

    summary["last_close"] = float(df["Close"].iloc[-1])

    # -----------------------------
    # INDICATORS
    # -----------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    summary["ema9"] = float(df["EMA9"].iloc[-1])
    summary["ema20"] = float(df["EMA20"].iloc[-1])
    summary["rsi"] = float(df["RSI"].iloc[-1])

    returns = df["Close"].pct_change().dropna()

    summary["recent_volatility"] = float(returns.tail(20).std())


    # -----------------------------
    # FORECAST LOOP
    # -----------------------------

    for horizon in forecast_horizons:

        assistant.reset()

        prompt = f"""
You are an AI intraday stock analyst.

Stock: {symbol}
Current Timestamp: {forward_timestamp}

Forecast Horizon: {horizon} minutes

Current Price: {summary['last_close']}

Technical Indicators
EMA9: {summary['ema9']}
EMA20: {summary['ema20']}
RSI14: {summary['rsi']}
Recent Volatility: {summary['recent_volatility']}

Predict the expected move over the next {horizon} minutes.

Return ONLY JSON:

{{
"symbol": "{symbol}",
"horizon_minutes": {horizon},
"current_price": {summary['last_close']},
"predicted_price": number,
"predicted_direction": "UP or DOWN",
"expected_move_percent": number
}}
"""

        response = assistant.chat(prompt)

        assistant_msg = str(response)

        # -----------------------------
        # PARSE RESPONSE
        # -----------------------------

        try:
            start = assistant_msg.index("{")
            end = assistant_msg.rindex("}") + 1
            json_text = assistant_msg[start:end]

            data = json.loads(json_text)

            predicted_price = float(data["predicted_price"])
            predicted_direction = data["predicted_direction"]
            predicted_move = float(data["expected_move_percent"])

        except Exception as e:
            print("JSON Parse Error:", e)
            print("Raw response:", assistant_msg)

            predicted_price = summary["last_close"]
            predicted_direction = "NEUTRAL"
            predicted_move = 0.0

        # -----------------------------
        # STORE RESULT
        # -----------------------------

        forward_results.append({

            "timestamp": forward_timestamp,
            "symbol": symbol,
            "horizon_minutes": horizon,
            "current_price": summary["last_close"],
            "predicted_price": predicted_price,
            "predicted_direction": predicted_direction,
            "expected_move_percent": predicted_move

        })


# -----------------------------
# FINAL TABLE
# -----------------------------

forward_df = pd.DataFrame(forward_results)

print("\nForward Test Predictions\n")

print(forward_df)

[autogen.oai.client: 03-10 10:00:52] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:00:52] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:00:52] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:00:52] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:00:52] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
Forward Test Timestamp: 2026-03-10 10:00:52

Running Forward Test: TCS.NS
User_Proxy (to Market_Analyst):


You are an AI intraday stock analyst.

Stock: TCS.NS
Current Timestamp: 2026-03-10 10:00:52

Forecast Horizon: 15 minutes

Current Price: 2510.6000

In [1]:
# ==========================================
# FINROBOT FORWARD TEST (IT STOCKS)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# FINROBOT CONFIG
llm_config = {
    "config_list": [{
        "model": "gpt-4o-mini",
        "api_key": os.environ["OPENAI_API_KEY"]
    }],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# IT STOCKS
stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS"
]

interval = "5m"

predictions = []


print("\nGenerating Predictions...\n")


# ======================================
# STEP 1 — PREDICT ALL STOCKS
# ======================================

for symbol in stocks:

    df = yf.download(symbol, period="2d", interval=interval, progress=False)

    current_price = float(df["Close"].iloc[-1])

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    ema9 = float(df["EMA9"].iloc[-1])
    ema20 = float(df["EMA20"].iloc[-1])

    prompt = f"""
Stock: {symbol}

Current Price: {current_price}

EMA9: {ema9}
EMA20: {ema20}

Will price go UP or DOWN in next 15 minutes?

Answer only UP or DOWN.
"""

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    output = buffer.getvalue()

    if "UP" in output.upper():
        direction = "UP"
        predicted_price = current_price * 1.002
    else:
        direction = "DOWN"
        predicted_price = current_price * 0.998

    predictions.append({
        "Stock": symbol,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Direction": direction
    })

    print(symbol, "Prediction:", direction)


# ======================================
# STEP 2 — WAIT 15 MINUTES
# ======================================

print("\nWaiting 15 minutes for actual prices...\n")

time.sleep(15 * 60)


# ======================================
# STEP 3 — GET ACTUAL PRICES
# ======================================

for p in predictions:

    df = yf.download(p["Stock"], period="1d", interval="5m", progress=False)

    actual_price = float(df["Close"].iloc[-1])

    error = abs(p["Predicted Price"] - actual_price) / actual_price * 100

    p["Actual Price"] = actual_price
    p["Error %"] = error


# ======================================
# STEP 4 — RESULTS
# ======================================

results_df = pd.DataFrame(predictions)

print("\n==============================")
print("FORWARD TEST RESULTS")
print("==============================\n")

print(results_df)

print("\nAverage Error %:", results_df["Error %"].mean())

[autogen.oai.client: 03-10 10:30:13] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:30:13] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:30:13] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:30:13] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 10:30:13] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating Predictions...



C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:52: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df["Close"].iloc[-1])


TCS.NS Prediction: UP


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:52: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df["Close"].iloc[-1])


INFY.NS Prediction: UP


...

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:52: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df["Close"].iloc[-1])


TECHM.NS Prediction: UP

Waiting 15 minutes for actual prices...



C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:116: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df["Close"].iloc[-1])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:116: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df["Close"].iloc[-1])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:116: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df["Close"].iloc[-1])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:116: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_


FORWARD TEST RESULTS

        Stock  Current Price  Predicted Price Direction  Actual Price  \
0      TCS.NS    2514.899902      2519.929702        UP   2518.000000   
1     INFY.NS    1299.000000      1301.598000        UP   1301.000000   
2  HCLTECH.NS    1351.699951      1354.403351        UP   1353.199951   
3    WIPRO.NS     198.600006       198.997206        UP    198.899994   
4    TECHM.NS    1326.900024      1329.553824        UP   1326.900024   

    Error %  
0  0.076636  
1  0.045965  
2  0.088930  
3  0.048875  
4  0.200000  

Average Error %: 0.09208116202893431


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_7432\1656633850.py:116: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df["Close"].iloc[-1])


In [1]:
# ==========================================
# FINROBOT FORWARD TEST (IT SECTOR)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# -----------------------------
# FINROBOT CONFIG
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# -----------------------------
# NSE IT STOCKS
# -----------------------------
stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS"
]

interval = "5m"

predictions = []

print("\nGenerating Predictions...\n")


# ======================================
# STEP 1 — PREDICT ALL STOCKS
# ======================================

for symbol in stocks:

    print("\nAnalyzing:", symbol)

    df = yf.download(symbol, period="2d", interval=interval, progress=False)

    if df.empty:
        continue

    current_price = df["Close"].iloc[-1].item()

    # -----------------------------
    # INDICATORS
    # -----------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    ema9 = df["EMA9"].iloc[-1].item()
    ema20 = df["EMA20"].iloc[-1].item()
    rsi = df["RSI"].iloc[-1].item()


    # ======================================
    # FINROBOT PROMPT
    # ======================================

    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}

Current Price: {current_price}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}

Will the price move UP or DOWN in the next 15 minutes?

Answer ONLY:
UP or DOWN
"""

    assistant.reset()

    # capture conversation but also print it
    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    output = buffer.getvalue()

    print(output)

    # -----------------------------
    # DETECT PREDICTION
    # -----------------------------

    if "UP" in output.upper():
        direction = "UP"
        predicted_price = current_price * 1.002

    elif "DOWN" in output.upper():
        direction = "DOWN"
        predicted_price = current_price * 0.998

    else:
        direction = "UNKNOWN"
        predicted_price = current_price


    predictions.append({
        "Stock": symbol,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Direction": direction
    })

    print("Detected Prediction:", direction)


# ======================================
# STEP 2 — WAIT 15 MINUTES
# ======================================

print("\nWaiting 15 minutes for actual prices...\n")

time.sleep(15 * 60)


# ======================================
# STEP 3 — GET ACTUAL PRICES
# ======================================

for p in predictions:

    df = yf.download(p["Stock"], period="1d", interval="5m", progress=False)

    actual_price = df["Close"].iloc[-1].item()

    error = abs(p["Predicted Price"] - actual_price) / actual_price * 100

    p["Actual Price"] = actual_price
    p["Error %"] = error


# ======================================
# STEP 4 — RESULTS
# ======================================

results_df = pd.DataFrame(predictions)

print("\n==============================")
print("FORWARD TEST RESULTS")
print("==============================\n")

print(results_df)

print("\nAverage Error %:", results_df["Error %"].mean())

[autogen.oai.client: 03-10 11:29:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 11:29:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 11:29:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 11:29:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 11:29:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating Predictions...


Analyzing: TCS.NS
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS

Current Price: 2509.800048828125

Indicators:
EMA9: 2513.6191510641074
EMA20: 2515.4977796995527
RSI: 28.78791615

In [1]:
# ==========================================
# FINROBOT FORWARD TEST (30 MINUTES)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# -----------------------------
# FINROBOT CONFIG
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# -----------------------------
# NSE IT STOCKS
# -----------------------------
stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS"
]

interval = "5m"

predictions = []

print("\nGenerating 30-Minute Predictions...\n")


# ======================================
# STEP 1 — PREDICT ALL STOCKS
# ======================================

for symbol in stocks:

    print("\nAnalyzing:", symbol)

    df = yf.download(symbol, period="2d", interval=interval, progress=False)

    if df.empty:
        continue

    current_price = df["Close"].iloc[-1].item()

    # -----------------------------
    # INDICATORS
    # -----------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    ema9 = df["EMA9"].iloc[-1].item()
    ema20 = df["EMA20"].iloc[-1].item()
    rsi = df["RSI"].iloc[-1].item()


    # ======================================
    # FINROBOT PROMPT
    # ======================================

    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}

Current Price: {current_price}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}

Will the price move UP or DOWN in the next 30 minutes?

Answer ONLY:
UP or DOWN
"""

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    output = buffer.getvalue()

    print(output)


    # -----------------------------
    # DETECT PREDICTION
    # -----------------------------

    if "UP" in output.upper():
        direction = "UP"
        predicted_price = current_price * 1.004   # slightly bigger move for 30 min

    elif "DOWN" in output.upper():
        direction = "DOWN"
        predicted_price = current_price * 0.996

    else:
        direction = "UNKNOWN"
        predicted_price = current_price


    predictions.append({
        "Stock": symbol,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Direction": direction
    })

    print("Detected Prediction:", direction)


# ======================================
# STEP 2 — WAIT 30 MINUTES
# ======================================

print("\nWaiting 30 minutes for actual prices...\n")

time.sleep(30 * 60)


# ======================================
# STEP 3 — GET ACTUAL PRICES
# ======================================

for p in predictions:

    df = yf.download(p["Stock"], period="1d", interval="5m", progress=False)

    actual_price = df["Close"].iloc[-1].item()

    error = abs(p["Predicted Price"] - actual_price) / actual_price * 100

    p["Actual Price"] = actual_price
    p["Error %"] = error


# ======================================
# STEP 4 — RESULTS
# ======================================

# ======================================
# STEP 4 — RESULTS
# ======================================

print("\n==============================")
print("30-MINUTE FORWARD TEST RESULTS")
print("==============================\n")

# Column headers with padding
print(
    f"{'Stock':<12}"
    f"{'Current Price':<18}"
    f"{'Predicted Price':<18}"
    f"{'Direction':<12}"
    f"{'Actual Price':<15}"
    f"{'Error %':<10}"
)


print("-" * 85)

errors = []

for p in predictions:

    error = p["Error %"]
    errors.append(error)

    print(
        f"{p['Stock']:<12}"
        f"{p['Current Price']:<18.2f}"
        f"{p['Predicted Price']:<18.2f}"
        f"{p['Direction']:<12}"
        f"{p['Actual Price']:<15.2f}"
        f"{error:<10.2f}"
    )

avg_error = sum(errors) / len(errors)

print("\nAverage Error %:", round(avg_error, 2))

[autogen.oai.client: 03-10 12:00:00] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:00:00] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:00:00] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:00:00] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:00:00] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating 30-Minute Predictions...


Analyzing: TCS.NS
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS

Current Price: 2503.699951171875

Indicators:
EMA9: 2508.2588127207414
EMA20: 2511.564394140082
RSI: 10

In [2]:
# ==========================================
# FINROBOT FORWARD TEST (1 HOUR)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# -----------------------------
# FINROBOT CONFIG
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# -----------------------------
# NSE IT STOCKS
# -----------------------------
stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS"
]

interval = "5m"

predictions = []

print("\nGenerating 1-Hour Predictions...\n")


# ======================================
# STEP 1 — PREDICT ALL STOCKS
# ======================================

for symbol in stocks:

    print("\nAnalyzing:", symbol)

    df = yf.download(symbol, period="2d", interval=interval, progress=False)

    if df.empty:
        continue

    current_price = round(df["Close"].iloc[-1].item(), 2)

    # -----------------------------
    # INDICATORS
    # -----------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    ema9 = round(df["EMA9"].iloc[-1].item(), 2)
    ema20 = round(df["EMA20"].iloc[-1].item(), 2)
    rsi = round(df["RSI"].iloc[-1].item(), 2)


    # ======================================
    # FINROBOT PROMPT
    # ======================================

    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}

Current Price: {current_price}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}

Will the price move UP or DOWN in the next 1 hour?

Answer ONLY:
UP or DOWN
"""

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    output = buffer.getvalue()

    print(output)

    # -----------------------------
    # DETECT PREDICTION
    # -----------------------------

    if "UP" in output.upper():
        direction = "UP"
        predicted_price = round(current_price * 1.008, 2)

    elif "DOWN" in output.upper():
        direction = "DOWN"
        predicted_price = round(current_price * 0.992, 2)

    else:
        direction = "UNKNOWN"
        predicted_price = current_price


    predictions.append({
        "Stock": symbol,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Direction": direction
    })

    print("Detected Prediction:", direction)


# ======================================
# STEP 2 — WAIT 1 HOUR
# ======================================

print("\nWaiting 1 hour for actual prices...\n")

time.sleep(60 * 60)


# ======================================
# STEP 3 — GET ACTUAL PRICES
# ======================================

for p in predictions:

    df = yf.download(p["Stock"], period="1d", interval="5m", progress=False)

    actual_price = round(df["Close"].iloc[-1].item(), 2)

    error = abs(p["Predicted Price"] - actual_price) / actual_price * 100

    p["Actual Price"] = actual_price
    p["Error %"] = round(error, 2)


# ======================================
# STEP 4 — RESULTS (WITH PADDING)
# ======================================

print("\n===================================")
print("1-HOUR FORWARD TEST RESULTS")
print("===================================\n")

print(
    f"{'Stock':<12}"
    f"{'Current Price':<18}"
    f"{'Predicted Price':<18}"
    f"{'Direction':<12}"
    f"{'Actual Price':<15}"
    f"{'Error %':<10}"
)

print("-" * 85)

errors = []

for p in predictions:

    errors.append(p["Error %"])

    print(
        f"{p['Stock']:<12}"
        f"{p['Current Price']:<18.2f}"
        f"{p['Predicted Price']:<18.2f}"
        f"{p['Direction']:<12}"
        f"{p['Actual Price']:<15.2f}"
        f"{p['Error %']:<10.2f}"
    )

avg_error = sum(errors) / len(errors)

print("\nAverage Error %:", round(avg_error, 2))

[autogen.oai.client: 03-10 12:30:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:30:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:30:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:30:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 12:30:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating 1-Hour Predictions...


Analyzing: TCS.NS
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS

Current Price: 2500.0

Indicators:
EMA9: 2501.84
EMA20: 2505.92
RSI: 19.67

Will the price move UP or DOWN

In [ ]:
# ==========================================
# FINROBOT FORWARD TEST (2 HOURS)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# -----------------------------
# FINROBOT CONFIG
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.environ["OPENAI_API_KEY"]
        }
    ],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# -----------------------------
# NSE IT STOCKS
# -----------------------------
stocks = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS"
]

interval = "5m"

predictions = []

print("\nGenerating 2-Hour Predictions...\n")


# ======================================
# STEP 1 — PREDICT ALL STOCKS
# ======================================

for symbol in stocks:

    print("\nAnalyzing:", symbol)

    df = yf.download(symbol, period="2d", interval=interval, progress=False)

    if df.empty:
        continue

    current_price = round(df["Close"].iloc[-1].item(), 2)

    # -----------------------------
    # INDICATORS
    # -----------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    ema9 = round(df["EMA9"].iloc[-1].item(), 2)
    ema20 = round(df["EMA20"].iloc[-1].item(), 2)
    rsi = round(df["RSI"].iloc[-1].item(), 2)


    # ======================================
    # FINROBOT PROMPT
    # ======================================

    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}

Current Price: {current_price}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}

Will the price move UP or DOWN in the next 2 hours?

Answer ONLY:
UP or DOWN
"""

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    output = buffer.getvalue()

    print(output)

    # -----------------------------
    # DETECT PREDICTION
    # -----------------------------

    if "UP" in output.upper():
        direction = "UP"
        predicted_price = round(current_price * 1.015, 2)

    elif "DOWN" in output.upper():
        direction = "DOWN"
        predicted_price = round(current_price * 0.985, 2)

    else:
        direction = "UNKNOWN"
        predicted_price = current_price


    predictions.append({
        "Stock": symbol,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Direction": direction
    })

    print("Detected Prediction:", direction)


# ======================================
# STEP 2 — WAIT 2 HOURS
# ======================================

print("\nWaiting 2 hours for actual prices...\n")

time.sleep(120 * 60)


# ======================================
# STEP 3 — GET ACTUAL PRICES
# ======================================

for p in predictions:

    df = yf.download(p["Stock"], period="1d", interval="5m", progress=False)

    actual_price = round(df["Close"].iloc[-1].item(), 2)

    error = abs(p["Predicted Price"] - actual_price) / actual_price * 100

    p["Actual Price"] = actual_price
    p["Error %"] = round(error, 2)


# ======================================
# STEP 4 — RESULTS (WITH PADDING)
# ======================================

print("\n===================================")
print("2-HOUR FORWARD TEST RESULTS")
print("===================================\n")

print(
    f"{'Stock':<12}"
    f"{'Current Price':<18}"
    f"{'Predicted Price':<18}"
    f"{'Direction':<12}"
    f"{'Actual Price':<15}"
    f"{'Error %':<10}"
)

print("-" * 85)

errors = []

for p in predictions:

    errors.append(p["Error %"])

    print(
        f"{p['Stock']:<12}"
        f"{p['Current Price']:<18.2f}"
        f"{p['Predicted Price']:<18.2f}"
        f"{p['Direction']:<12}"
        f"{p['Actual Price']:<15.2f}"
        f"{p['Error %']:<10.2f}"
    )

avg_error = sum(errors) / len(errors)

print("\nAverage Error %:", round(avg_error, 2))

[autogen.oai.client: 03-10 13:30:25] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 13:30:25] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 13:30:25] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 13:30:25] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-10 13:30:25] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating 2-Hour Predictions...


Analyzing: TCS.NS
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS

Current Price: 2497.2

Indicators:
EMA9: 2495.37
EMA20: 2497.88
RSI: 48.98

Will the price move UP or DOWN

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
from finrobot.agents.workflow import SingleAssistant

# Minimal config
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    ],
    "timeout": 60,
}

# Create agent
assistant = SingleAssistant(
    {"name": "Market_Analyst", "description": "Stock analyst"},
    llm_config,
    human_input_mode="NEVER"
)

print("FinRobot agent initialized")

[autogen.oai.client: 03-10 19:06:53] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
FinRobot agent initialized


In [3]:
# ==========================================
# FINROBOT LLM INTRADAY BACKTEST (IT SECTOR)
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------------
# IT SECTOR STOCKS
# -----------------------------
symbols = [
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "TECHM.NS",
    "LTIM.NS",
    "MPHASIS.NS",
    "PERSISTENT.NS",
    "COFORGE.NS",
    "SONATSOFTW.NS"
]

# -----------------------------
# SAME TIMESTAMP FOR ALL
# -----------------------------
timestamp = "2026-02-25 14:00:00"

target_datetime = pd.Timestamp(timestamp, tz="Asia/Kolkata")

# Store results
results = []

# -----------------------------
# LOOP THROUGH STOCKS
# -----------------------------
for symbol in symbols:

    print(f"\nProcessing {symbol} at {target_datetime}")

    # -----------------------------
    # DOWNLOAD DATA
    # -----------------------------
    df = yf.download(symbol, period="60d", interval="15m", progress=False)

    if df.empty:
        print("No data")
        continue

    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert("Asia/Kolkata")

    if target_datetime not in df.index:
        print("Timestamp not found")
        continue

    # -----------------------------
    # HISTORY
    # -----------------------------
    history_df = df[df.index < target_datetime].tail(7 * 25)

    current_price = float(df.loc[target_datetime]["Close"])

    history_text = history_df[
        ["Open", "High", "Low", "Close", "Volume"]
    ].to_string()

    # -----------------------------
    # PROMPT
    # -----------------------------
    prompt = f"""
You are a professional NSE intraday analyst.

Stock: {symbol}
Prediction Time: {target_datetime}

Past 7 days 15-minute OHLCV data:

{history_text}

Predict whether NEXT candle will move UP or DOWN.

Respond:
Prediction: UP or DOWN
Reason: short explanation
"""

    response = assistant.chat(prompt)

    # -----------------------------
    # PREDICTION
    # -----------------------------
    if response and "UP" in response.upper():
        predicted_direction = "UP"

    elif response and "DOWN" in response.upper():
        predicted_direction = "DOWN"

    else:
        last_close = float(history_df["Close"].iloc[-1])
        prev_close = float(history_df["Close"].iloc[-2])
        predicted_direction = "UP" if last_close > prev_close else "DOWN"

    predicted_price = (
        current_price * 1.002
        if predicted_direction == "UP"
        else current_price * 0.998
    )

    # -----------------------------
    # ACTUAL RESULT
    # -----------------------------
    next_index = df.index.get_loc(target_datetime) + 1

    if next_index >= len(df):
        continue

    actual_price = float(df.iloc[next_index]["Close"])

    actual_direction = "UP" if actual_price > current_price else "DOWN"

    # -----------------------------
    # ERROR %
    # -----------------------------
    error_percentage = abs(predicted_price - actual_price) / actual_price * 100

    results.append({
        "Stock": symbol,
        "Time": target_datetime,
        "Curr Price": current_price,
        "Pred Price": predicted_price,
        "Act Price": actual_price,
        "Pred Dir": predicted_direction,
        "Act Dir": actual_direction,
        "Error %": error_percentage
    })

# -----------------------------
# RESULT TABLE
# -----------------------------
results_df = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

print("\n====================================")
print("FINAL BACKTEST RESULTS")
print("====================================")

print(results_df)

# -----------------------------
# AVERAGE ERROR
# -----------------------------
if not results_df.empty:
    print("\nAverage Error %:", results_df["Error %"].mean())


Processing TCS.NS at 2026-02-25 14:00:00+05:30
User_Proxy (to Market_Analyst):


You are a professional NSE intraday analyst.

Stock: TCS.NS
Prediction Time: 2026-02-25 14:00:00+05:30

Past 7 days 15-minute OHLCV data:

Price                             Open         High          Low        Close  Volume
Ticker                          TCS.NS       TCS.NS       TCS.NS       TCS.NS  TCS.NS
Datetime                                                                             
2026-02-16 14:00:00+05:30  2684.800049  2687.300049  2681.000000  2684.899902  124419
2026-02-16 14:15:00+05:30  2684.899902  2699.899902  2684.000000  2698.300049   96498
2026-02-16 14:30:00+05:30  2698.199951  2706.899902  2695.500000  2699.600098  158228
2026-02-16 14:45:00+05:30  2699.600098  2699.600098  2693.000000  2698.000000   78296
2026-02-16 15:00:00+05:30  2697.399902  2708.500000  2697.399902  2707.000000  181673
2026-02-16 15:15:00+05:30  2707.000000  2716.899902  2706.100098  2713.500000  159519
2026-

In [1]:
# ==========================================
# FINROBOT AI FORWARD TEST (FINAL STABLE)
# ==========================================

import os
import time
import io
import contextlib
import yfinance as yf
import pandas as pd

from finrobot.agents.workflow import SingleAssistant


# ---------------------------------
# FINROBOT CONFIG
# ---------------------------------

llm_config = {
    "config_list": [{
        "model": "gpt-4o-mini",
        "api_key": os.environ["OPENAI_API_KEY"]
    }],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# ---------------------------------
# STOCK LIST
# ---------------------------------

stocks = [
    "TCS.NS","INFY.NS","HCLTECH.NS","WIPRO.NS","TECHM.NS",
    "MPHASIS.NS","PERSISTENT.NS","COFORGE.NS","SONATSOFTW.NS",
    "ZENSARTECH.NS","BSOFT.NS","KPITTECH.NS","TATAELXSI.NS"
]

interval = "5m"

predictions = []

print("\nGenerating AI Predictions...\n")


# ======================================
# SAFE DOWNLOAD FUNCTION
# ======================================

def safe_download(symbol):

    for attempt in range(3):

        try:

            df = yf.download(
                symbol,
                period="7d",
                interval="5m",
                progress=False,
                threads=False
            )

            if df is None or len(df) == 0:
                time.sleep(2)
                continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df = df.dropna()

            return df

        except Exception:
            time.sleep(2)

    print("Skipping", symbol, "- data unavailable")
    return None


# ======================================
# STEP 1 — PREDICTIONS
# ======================================

for symbol in stocks:

    print("\n====================================")
    print("Analyzing:", symbol)
    print("====================================")

    df = safe_download(symbol)

    if df is None:
        continue

    current_price = float(df["Close"].iloc[-1])

    # ---------------------------------
    # INDICATORS
    # ---------------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    df["VWAP"] = (tp * df["Volume"]).cumsum() / df["Volume"].cumsum()

    df["VOL_MA"] = df["Volume"].rolling(20).mean()

    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = abs(df["High"] - df["Close"].shift())
    df["L-PC"] = abs(df["Low"] - df["Close"].shift())

    df["TR"] = df[["H-L","H-PC","L-PC"]].max(axis=1)
    df["ATR"] = df["TR"].rolling(14).mean()

    ema9 = float(df["EMA9"].iloc[-1])
    ema20 = float(df["EMA20"].iloc[-1])
    rsi = float(df["RSI"].iloc[-1])
    vwap = float(df["VWAP"].iloc[-1])
    vol = float(df["Volume"].iloc[-1])
    vol_ma = float(df["VOL_MA"].iloc[-1])
    atr = float(df["ATR"].iloc[-1])


    # ---------------------------------
    # FINROBOT PROMPT
    # ---------------------------------

    prompt = f"""
You are an expert NSE intraday analyst.

Stock: {symbol}

Current Price: {current_price}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}
VWAP: {vwap}
Volume: {vol}
Volume MA20: {vol_ma}
ATR: {atr}

Explain briefly your reasoning.

Then return ONLY predictions exactly like this:

15M: UP/DOWN/NEUTRAL
30M: UP/DOWN/NEUTRAL
1H: UP/DOWN/NEUTRAL
"""


    # ---------------------------------
    # CAPTURE AGENT CONVERSATION
    # ---------------------------------

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    response = buffer.getvalue()

    print("\nAgent Analysis:\n")
    print(response)

    output = response.upper()


    # ---------------------------------
    # PARSE PREDICTIONS
    # ---------------------------------

    def extract(label):

        if label in output:

            part = output.split(label)[1].split("\n")[0]

            if "UP" in part:
                return "UP"

            if "DOWN" in part:
                return "DOWN"

        return "NEUTRAL"


    predictions.append({

        "Stock": symbol,
        "Current": current_price,
        "15M": extract("15M"),
        "30M": extract("30M"),
        "1H": extract("1H")

    })


# ======================================
# STEP 2 — WAIT
# ======================================

print("\nWaiting 15 minutes...\n")
time.sleep(15*60)


future_prices = {}

for s in stocks:

    df = safe_download(s)

    if df is not None:
        future_prices[s] = float(df["Close"].iloc[-1])


for p in predictions:

    if p["Stock"] in future_prices:
        p["Actual15"] = future_prices[p["Stock"]]


print("\nWaiting another 15 minutes...\n")
time.sleep(15*60)


future_prices = {}

for s in stocks:

    df = safe_download(s)

    if df is not None:
        future_prices[s] = float(df["Close"].iloc[-1])


for p in predictions:

    if p["Stock"] in future_prices:
        p["Actual30"] = future_prices[p["Stock"]]


print("\nWaiting another 30 minutes...\n")
time.sleep(30*60)


future_prices = {}

for s in stocks:

    df = safe_download(s)

    if df is not None:
        future_prices[s] = float(df["Close"].iloc[-1])


for p in predictions:

    if p["Stock"] in future_prices:
        p["Actual60"] = future_prices[p["Stock"]]


# ======================================
# STEP 3 — DIRECTION
# ======================================

def direction(curr, future):

    change = (future - curr) / curr * 100

    if change > 0.15:
        return "UP"

    if change < -0.15:
        return "DOWN"

    return "NEUTRAL"


correct15 = correct30 = correct60 = 0


for p in predictions:

    if "Actual15" not in p:
        continue

    p["Dir15"] = direction(p["Current"], p["Actual15"])
    p["Dir30"] = direction(p["Current"], p["Actual30"])
    p["Dir60"] = direction(p["Current"], p["Actual60"])

    if p["Dir15"] == p["15M"]:
        correct15 += 1

    if p["Dir30"] == p["30M"]:
        correct30 += 1

    if p["Dir60"] == p["1H"]:
        correct60 += 1


# ======================================
# RESULTS
# ======================================

print("\n==============================================")
print("FORWARD TEST RESULTS")
print("==============================================\n")

print(f"{'Stock':<15}{'15M Pred':<10}{'15M Act':<10}{'30M Pred':<10}{'30M Act':<10}{'1H Pred':<10}{'1H Act':<10}")

print("-"*85)

for p in predictions:

    if "Dir15" not in p:
        continue

    print(f"{p['Stock']:<15}{p['15M']:<10}{p['Dir15']:<10}{p['30M']:<10}{p['Dir30']:<10}{p['1H']:<10}{p['Dir60']:<10}")


n = len(predictions)

print("\nDirectional Accuracy")

print(f"15 MIN : {correct15/n*100:.2f}%")
print(f"30 MIN : {correct30/n*100:.2f}%")
print(f"1 HOUR : {correct60/n*100:.2f}%")

[autogen.oai.client: 03-11 10:18:28] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 10:18:28] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 10:18:28] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 10:18:28] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 10:18:28] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Generating AI Predictions...


Analyzing: TCS.NS

Agent Analysis:

User_Proxy (to Market_Analyst):


You are an expert NSE intraday analyst.

Stock: TCS.NS

Current Price: 2487.39990234375

Indicators:
EMA9: 2491.4288850137596
EMA20: 2496.7700727702
RSI:

In [3]:
# ==========================================
# FINROBOT LLM INTRADAY BACKTEST (MULTI STOCK)
# DATE: 27-02-2026
# ==========================================

import os
import yfinance as yf
import pandas as pd
import io
import contextlib

from finrobot.agents.workflow import SingleAssistant


# ---------------------------------
# FINROBOT CONFIG
# ---------------------------------

llm_config = {
    "config_list": [{
        "model": "gpt-4o-mini",
        "api_key": os.environ["OPENAI_API_KEY"]
    }],
    "temperature": 0
}

assistant = SingleAssistant("Market_Analyst", llm_config)


# ---------------------------------
# STOCK LIST
# ---------------------------------

symbols = [
"TCS.NS","INFY.NS","HCLTECH.NS","WIPRO.NS",
"TECHM.NS","LTIM.NS","MPHASIS.NS",
"PERSISTENT.NS","COFORGE.NS"
]


# ---------------------------------
# TIMESTAMPS (27-02-2026)
# ---------------------------------

timestamps = [
"2026-02-27 10:00:00",
"2026-02-27 10:15:00",
"2026-02-27 10:30:00",
"2026-02-27 10:45:00",
"2026-02-27 11:00:00",
"2026-02-27 11:15:00",
"2026-02-27 11:30:00",
"2026-02-27 11:45:00",
"2026-02-27 12:00:00"
]


results = []


# ==========================================
# LOOP OVER STOCKS
# ==========================================

for symbol, ts in zip(symbols, timestamps):

    target_datetime = pd.Timestamp(ts, tz="Asia/Kolkata")

    print("\n========================================")
    print("Processing:", symbol, "at", target_datetime)
    print("========================================")


    # ---------------------------------
    # DOWNLOAD DATA
    # ---------------------------------

    df = yf.download(symbol, period="60d", interval="15m", progress=False)

    if df.empty:
        print("No data")
        continue


    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert("Asia/Kolkata")


    if target_datetime not in df.index:
        print("Timestamp not found")
        continue


    # ---------------------------------
    # INDICATORS
    # ---------------------------------

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100/(1+rs))

    df["ATR"] = (df["High"] - df["Low"]).rolling(14).mean()


    # ---------------------------------
    # HISTORY
    # ---------------------------------

    history_df = df[df.index < target_datetime].tail(7*25)

    current_price = float(df.loc[target_datetime]["Close"])

    ema9 = float(df.loc[target_datetime]["EMA9"])
    ema20 = float(df.loc[target_datetime]["EMA20"])
    rsi = float(df.loc[target_datetime]["RSI"])
    atr = float(df.loc[target_datetime]["ATR"])


    history_text = history_df[
        ["Open","High","Low","Close","Volume"]
    ].tail(20).to_string()


    # ---------------------------------
    # FINROBOT PROMPT
    # ---------------------------------

    prompt = f"""
You are a professional NSE intraday trading analyst.

Stock: {symbol}
Time: {target_datetime}

Indicators:
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}
ATR: {atr}

Recent Market Data:

{history_text}

Rules:

EMA9 > EMA20 → bullish
EMA9 < EMA20 → bearish

RSI > 60 → strong bullish momentum
RSI < 40 → bearish momentum

Predict next candle direction.

Respond EXACTLY:

Prediction: UP or DOWN
Reason: short explanation
"""


    # ---------------------------------
    # CAPTURE AGENT CONVERSATION
    # ---------------------------------

    assistant.reset()

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        assistant.chat(prompt)

    response = buffer.getvalue()

    print("\nAgent Conversation:\n")
    print(response)


    # ---------------------------------
    # PARSE PREDICTION
    # ---------------------------------

    text = response.upper()

    if "UP" in text:
        predicted_direction = "UP"

    elif "DOWN" in text:
        predicted_direction = "DOWN"

    else:
        predicted_direction = "UP" if ema9 > ema20 else "DOWN"


    # ---------------------------------
    # PREDICTED PRICE
    # ---------------------------------

    if predicted_direction == "UP":
        predicted_price = current_price + atr*0.5
    else:
        predicted_price = current_price - atr*0.5


    # ---------------------------------
    # ACTUAL RESULT
    # ---------------------------------

    next_index = df.index.get_loc(target_datetime) + 1

    if next_index >= len(df):
        continue

    actual_price = float(df.iloc[next_index]["Close"])

    actual_direction = "UP" if actual_price > current_price else "DOWN"


    # ---------------------------------
    # ERROR %
    # ---------------------------------

    error_percentage = abs(predicted_price - actual_price)/actual_price*100


    results.append({
        "Stock": symbol,
        "Time": target_datetime,
        "Curr Price": current_price,
        "Pred Price": predicted_price,
        "Act Price": actual_price,
        "Pred Dir": predicted_direction,
        "Act Dir": actual_direction,
        "Error %": error_percentage
    })


# ==========================================
# RESULTS TABLE
# ==========================================

results_df = pd.DataFrame(results)

print("\n==============================================")
print("FINAL BACKTEST RESULTS")
print("==============================================\n")

print(f"{'Stock':<15}{'Time':<25}{'Curr':<10}{'Pred':<10}{'Actual':<10}{'PDir':<8}{'ADir':<8}{'Error %':<10}")
print("-"*95)

for _, r in results_df.iterrows():

    print(f"{r['Stock']:<15}{str(r['Time']):<25}{r['Curr Price']:<10.2f}{r['Pred Price']:<10.2f}{r['Act Price']:<10.2f}{r['Pred Dir']:<8}{r['Act Dir']:<8}{r['Error %']:<10.3f}")


# ==========================================
# METRICS
# ==========================================

if not results_df.empty:

    avg_error = results_df["Error %"].mean()

    correct = (results_df["Pred Dir"] == results_df["Act Dir"]).sum()

    total = len(results_df)

    directional_accuracy = correct/total*100

    print("\nAverage Error % :", avg_error)
    print("Directional Accuracy :", directional_accuracy,"%")

[autogen.oai.client: 03-11 11:46:18] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 11:46:18] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 11:46:18] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 11:46:18] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-11 11:46:18] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

Processing: TCS.NS at 2026-02-27 10:00:00+05:30


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:124: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df.loc[target_datetime]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:126: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ema9 = float(df.loc[target_datetime]["EMA9"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:127: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ema20 = float(df.loc[target_datetime]["EMA20"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:128: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.i


Agent Conversation:

User_Proxy (to Market_Analyst):


You are a professional NSE intraday trading analyst.

Stock: TCS.NS
Time: 2026-02-27 10:00:00+05:30

Indicators:
EMA9: 2652.2566864388496
EMA20: 2649.1380443983144
RSI: 60.90944401287316
ATR: 8.714268275669642

Recent Market Data:

Price                             Open         High          Low        Close  Volume
Ticker                          TCS.NS       TCS.NS       TCS.NS       TCS.NS  TCS.NS
Datetime                                                                             
2026-02-26 11:15:00+05:30  2653.500000  2658.699951  2650.800049  2653.000000   52110
2026-02-26 11:30:00+05:30  2653.000000  2653.899902  2641.899902  2646.000000  273137
2026-02-26 11:45:00+05:30  2645.600098  2652.000000  2645.399902  2648.199951  163327
2026-02-26 12:00:00+05:30  2648.100098  2655.199951  2648.100098  2651.800049   65103
2026-02-26 12:15:00+05:30  2651.800049  2655.000000  2642.699951  2647.600098  164081
2026-02-26 12:30:00+05:3

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:226: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df.iloc[next_index]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:124: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df.loc[target_datetime]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:126: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ema9 = float(df.loc[target_datetime]["EMA9"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:127: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(se


Agent Conversation:

User_Proxy (to Market_Analyst):


You are a professional NSE intraday trading analyst.

Stock: INFY.NS
Time: 2026-02-27 10:15:00+05:30

Indicators:
EMA9: 1300.9985501052133
EMA20: 1298.1171280922376
RSI: 64.58322736961838
ATR: 5.957162039620536

Recent Market Data:

Price                             Open         High          Low        Close  Volume
Ticker                         INFY.NS      INFY.NS      INFY.NS      INFY.NS INFY.NS
Datetime                                                                             
2026-02-26 11:30:00+05:30  1298.500000  1299.099976  1294.400024  1296.400024  262552
2026-02-26 11:45:00+05:30  1296.400024  1299.500000  1295.099976  1295.300049  282215
2026-02-26 12:00:00+05:30  1295.300049  1299.699951  1295.000000  1298.000000  327808
2026-02-26 12:15:00+05:30  1298.199951  1299.300049  1292.800049  1293.199951  338637
2026-02-26 12:30:00+05:30  1293.000000  1294.199951  1291.599976  1292.300049  364197
2026-02-26 12:45:00+05:

...

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:226: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df.iloc[next_index]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:124: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df.loc[target_datetime]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:126: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ema9 = float(df.loc[target_datetime]["EMA9"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:127: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(se


Agent Conversation:

User_Proxy (to Market_Analyst):


You are a professional NSE intraday trading analyst.

Stock: PERSISTENT.NS
Time: 2026-02-27 11:45:00+05:30

Indicators:
EMA9: 4798.823036035508
EMA20: 4795.759806466465
RSI: 53.61305361305361
ATR: 38.035714285714285

Recent Market Data:

Price                              Open          High           Low         Close        Volume
Ticker                    PERSISTENT.NS PERSISTENT.NS PERSISTENT.NS PERSISTENT.NS PERSISTENT.NS
Datetime                                                                                       
2026-02-26 13:00:00+05:30        4780.0        4789.5        4771.0        4776.5         13702
2026-02-26 13:15:00+05:30        4776.5        4786.0        4764.0        4780.0         16414
2026-02-26 13:30:00+05:30        4780.0        4786.0        4760.5        4775.0         18844
2026-02-26 13:45:00+05:30        4775.0        4792.5        4768.5        4785.5         14722
2026-02-26 14:00:00+05:30        4

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:226: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df.iloc[next_index]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:124: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  current_price = float(df.loc[target_datetime]["Close"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:126: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ema9 = float(df.loc[target_datetime]["EMA9"])
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:127: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(se


Agent Conversation:

User_Proxy (to Market_Analyst):


You are a professional NSE intraday trading analyst.

Stock: COFORGE.NS
Time: 2026-02-27 12:00:00+05:30

Indicators:
EMA9: 1203.696683461842
EMA20: 1205.4289197990736
RSI: 45.36294837876084
ATR: 6.907148088727679

Recent Market Data:

Price                             Open         High          Low        Close     Volume
Ticker                      COFORGE.NS   COFORGE.NS   COFORGE.NS   COFORGE.NS COFORGE.NS
Datetime                                                                                
2026-02-26 13:15:00+05:30  1200.500000  1204.900024  1198.599976  1203.300049     110350
2026-02-26 13:30:00+05:30  1203.300049  1205.500000  1202.199951  1205.000000      96058
2026-02-26 13:45:00+05:30  1205.000000  1205.400024  1202.699951  1204.400024      87818
2026-02-26 14:00:00+05:30  1204.599976  1205.300049  1200.400024  1203.800049     119320
2026-02-26 14:15:00+05:30  1203.400024  1204.900024  1200.599976  1204.599976      739

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_23692\1820085466.py:226: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual_price = float(df.iloc[next_index]["Close"])


In [4]:
# ==========================================
# FINROBOT AI FORWARD TEST — 30M + 1H
# ==========================================

import os
import time
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

stocks = [
"HDFCBANK.NS","ICICIBANK.NS","KOTAKBANK.NS","AXISBANK.NS","SBIN.NS",
"RELIANCE.NS","ONGC.NS","NTPC.NS","POWERGRID.NS","COALINDIA.NS",
"TCS.NS","INFY.NS","WIPRO.NS","HCLTECH.NS","TECHM.NS",
"SUNPHARMA.NS","DRREDDY.NS","CIPLA.NS","APOLLOHOSP.NS","LUPIN.NS"
]

predictions = []

print("\nGenerating AI Predictions...\n")

# ======================================
# SAFE DOWNLOAD
# ======================================

def safe_download(symbol):

    end_date = datetime.today() + timedelta(days=1)
    start_date = datetime.today() - timedelta(days=5)

    for attempt in range(3):

        try:

            df = yf.download(
                symbol,
                start=start_date.strftime("%Y-%m-%d"),
                end=end_date.strftime("%Y-%m-%d"),
                interval="5m",
                progress=False,
                threads=False
            )

            if df is None or len(df) == 0:
                time.sleep(2)
                continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df = df.dropna()

            return df

        except:
            time.sleep(2)

    return None


# ======================================
# GPT CALL
# ======================================

def ask_gpt(prompt):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system","content":"You are an expert NSE intraday analyst."},
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content


# ======================================
# PRICE DIRECTION
# ======================================

def direction(curr,future):

    change = (future - curr) / curr * 100

    if change > 0.15:
        return "UP"

    if change < -0.15:
        return "DOWN"

    return "NEUTRAL"


# ======================================
# PRINT RESULTS
# ======================================

def print_results(timeframe,pred_key,actual_key):

    print("\n==============================")
    print("RESULTS:",timeframe)
    print("==============================")

    correct = 0
    valid = 0

    for p in predictions:

        if actual_key not in p:
            continue

        actual_dir = direction(p["Current"],p[actual_key])

        valid += 1

        if p[pred_key] == actual_dir:
            correct += 1
            match = "✓"
        else:
            match = "✗"

        print(p["Stock"],p[pred_key],"vs",actual_dir,match)

    if valid > 0:
        print("\nAccuracy:",correct,"/",valid,"(",round(correct/valid*100,2),"%)")


# ======================================
# STEP 1 — GENERATE PREDICTIONS
# ======================================

for symbol in stocks:

    print("\nAnalyzing:",symbol)

    df = safe_download(symbol)

    if df is None:
        continue

    current = float(df["Close"].iloc[-1])

    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    rs = gain.rolling(14).mean() / loss.rolling(14).mean()

    df["RSI"] = 100 - (100/(1+rs))

    ema9 = float(df["EMA9"].iloc[-1])
    ema20 = float(df["EMA20"].iloc[-1])
    rsi = float(df["RSI"].iloc[-1])

    prompt = f"""
Stock: {symbol}
Price: {current}
EMA9: {ema9}
EMA20: {ema20}
RSI: {rsi}

Predict short term direction.

Return ONLY:

30M: UP/DOWN/NEUTRAL
1H: UP/DOWN/NEUTRAL
"""

    response = ask_gpt(prompt)

    print(response)

    txt = response.upper()

    def extract(label):

        if label in txt:

            part = txt.split(label)[1].split("\n")[0]

            if "UP" in part:
                return "UP"

            if "DOWN" in part:
                return "DOWN"

        return "NEUTRAL"

    predictions.append({
        "Stock":symbol,
        "Current":current,
        "30M":extract("30M:"),
        "1H":extract("1H:")
    })


# ======================================
# WAIT 30 MIN → CHECK 30M
# ======================================

print("\nWaiting 30 minutes...\n")
time.sleep(30*60)

future_prices = {}

for s in stocks:

    df = safe_download(s)

    if df is not None:
        future_prices[s] = float(df["Close"].iloc[-1])

for p in predictions:

    if p["Stock"] in future_prices:
        p["Actual_30M"] = future_prices[p["Stock"]]

print_results("30M","30M","Actual_30M")


# ======================================
# WAIT ANOTHER 30 MIN → CHECK 1H
# ======================================

print("\nWaiting another 30 minutes...\n")
time.sleep(30*60)

future_prices = {}

for s in stocks:

    df = safe_download(s)

    if df is not None:
        future_prices[s] = float(df["Close"].iloc[-1])

for p in predictions:

    if p["Stock"] in future_prices:
        p["Actual_1H"] = future_prices[p["Stock"]]

print_results("1H","1H","Actual_1H")


# ======================================
# FINAL SUMMARY
# ======================================

print("\nFINAL SUMMARY\n")

for p in predictions:

    print(
        p["Stock"],
        "30M:",p["30M"],"vs",p.get("Actual_30M","-"),
        "1H:",p["1H"],"vs",p.get("Actual_1H","-")
    )


Generating AI Predictions...


Analyzing: HDFCBANK.NS
30M: UP  
1H: NEUTRAL

Analyzing: ICICIBANK.NS
30M: DOWN  
1H: DOWN

Analyzing: KOTAKBANK.NS
30M: DOWN  
1H: DOWN  

Analyzing: AXISBANK.NS
30M: DOWN  
1H: DOWN

Analyzing: SBIN.NS
30M: DOWN  
1H: DOWN  

Analyzing: RELIANCE.NS
30M: UP  
1H: NEUTRAL

Analyzing: ONGC.NS
30M: NEUTRAL  
1H: NEUTRAL

Analyzing: NTPC.NS
30M: DOWN  
1H: DOWN

Analyzing: POWERGRID.NS
30M: DOWN  
1H: DOWN

Analyzing: COALINDIA.NS
30M: NEUTRAL  
1H: UP

Analyzing: TCS.NS
30M: DOWN  
1H: DOWN

Analyzing: INFY.NS
30M: DOWN  
1H: DOWN

Analyzing: WIPRO.NS
30M: DOWN  
1H: DOWN  

Analyzing: HCLTECH.NS
30M: DOWN  
1H: DOWN

Analyzing: TECHM.NS
30M: UP  
1H: NEUTRAL

Analyzing: SUNPHARMA.NS
30M: DOWN  
1H: DOWN

Analyzing: DRREDDY.NS
30M: DOWN  
1H: DOWN

Analyzing: CIPLA.NS
30M: DOWN  
1H: DOWN

Analyzing: APOLLOHOSP.NS
30M: DOWN  
1H: DOWN

Analyzing: LUPIN.NS
30M: DOWN  
1H: DOWN

Waiting 30 minutes...


RESULTS: 30M
HDFCBANK.NS UP vs DOWN ✗
ICICIBANK.NS DOWN 